# 02 - Fixed Mass Components

The total aircraft mass is the sum of **calculated masses** (wing structure, fuselage, battery, propulsion, which are estimated by parametric equations) and **fixed masses** (subsystems specified directly in the JSON).

This notebook examines the fixed mass inputs and their role in the overall weight budget. Understanding which masses are fixed vs. calculated is important because only the calculated masses change during MTOW iteration.

In [ ]:
import sys
sys.path.append('../../evtol')
from aircraft import Aircraft

aircraft = Aircraft('../Archer_Midnight.json')

: 

## Fixed Mass Inputs

These masses are specified directly in the JSON file and are **not derived** from other parameters. These values typically come reference aircraft data or engineering estimates. They remain constant throughout the MTOW iteration process.

In [ ]:
fixed_masses = {
    'Payload':                aircraft.payload_kg,
    'Actuators':              aircraft.actuator_mass_kg,
    'Furnishings':            aircraft.furnishings_mass_kg,
    'Env. Control System':    aircraft.environmental_control_system_mass_kg,
    'Avionics':               aircraft.avionics_mass_kg,
    'Hi-Volt Power Dist.':    aircraft.hivolt_power_dist_mass_kg,
    'Lo-Volt Power & Comms':  aircraft.lovolt_power_coms_mass_kg,
}

total_fixed = sum(fixed_masses.values())

print(f"{'Component':<25s} {'Mass (kg)':>10s}")
print("-" * 36)
for name, mass in fixed_masses.items():
    print(f"{name:<25s} {mass:>10.1f}")
print("-" * 36)
print(f"{'Total Fixed Mass':<25s} {total_fixed:>10.1f}")
print(f"{'% of MTOW':<25s} {100*total_fixed/aircraft.max_takeoff_mass_kg:>9.1f}%")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart of fixed masses
names = list(fixed_masses.keys())
masses = list(fixed_masses.values())
colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f', '#edc948', '#b07aa1']
wedges, texts, autotexts = axes[0].pie(masses, labels=names, autopct='%1.1f%%',
                                        colors=colors, pctdistance=0.8,
                                        textprops={'fontsize': 9})
axes[0].set_title('Fixed Mass Components')

# Bar chart: fixed vs remaining MTOW
remaining = aircraft.max_takeoff_mass_kg - total_fixed
categories = ['Payload', 'Other Fixed\nSubsystems', 'Calculated +\nMargin']
cat_values = [aircraft.payload_kg,
              total_fixed - aircraft.payload_kg,
              remaining]
cat_colors = ['#4e79a7', '#f28e2b', '#76b7b2']
bars = axes[1].bar(categories, cat_values, color=cat_colors, edgecolor='white')
for bar, val in zip(bars, cat_values):
    pct = 100 * val / aircraft.max_takeoff_mass_kg
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                 f'{val:.0f} kg\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
axes[1].set_ylabel('Mass (kg)')
axes[1].set_title(f'Fixed vs Calculated Mass (MTOW = {aircraft.max_takeoff_mass_kg:.0f} kg)')

plt.tight_layout()
plt.show()

## Mass Margin

The `mass_margin_factor` adds a percentage-based margin to the calculated empty weight to account for uncertainties in early design. This is standard in aircraft conceptual design:

The margin applies only to the calculated (non-payload) mass, so it amplifies the effect of any mass growth during the covergence process.

In [ ]:
print(f"Mass Margin Factor: {aircraft.mass_margin_factor:.2%}")
print(f"  This adds {aircraft.mass_margin_factor:.0%} to the calculated empty weight.")

## Summary

- **Fixed masses** (payload, avionics, actuators, ECS, power distribution, furnishings) are set directly in the JSON and do not change during MTOW iteration.
- **Payload** is the largest fixed component and represents the design requirement the aircraft must satisfy.
- The remaining fixed subsystems are relatively small individually but add up to a meaningful fraction of MTOW.
- The **mass margin** applies a percentage factor to account for design uncertainty. Even a small margin percentage can translate to significant mass given the snowball effect in aircraft sizing (more mass -> more power -> more battery -> more mass).

**Next:** Move on to [03 - Mission Profiles] to understand the flight segments.